# Sommeil EOG — CNN + Bi-LSTM (Kaggle)

**Mode reprise** : charge `sleep_model_v1_best.keras` et continue jusqu'a l'epoch 40.

Dataset requis (4 fichiers) : npz + 2 json + `sleep_model_v1_best.keras`

In [ ]:
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
)
from sklearn.utils import class_weight
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import (
    BatchNormalization,
    Bidirectional,
    Conv1D,
    Dense,
    Dropout,
    LSTM,
    MaxPooling1D,
)
from tensorflow.keras.models import Sequential

SEED = 42
EPOCHS = 40
INITIAL_EPOCH = 25
RESUME = True
BATCH_SIZE = 64
PATIENCE = 7
N_SAMPLES = 3000
STAGE_NAMES = ["W", "N1", "N2", "N3", "REM"]
CORPUS_FILES = ("sleep_edf_corpus.npz", "sleep_edf_corpus_meta.json", "subject_split.json")
CHECKPOINT_NAME = "sleep_model_v1_best.keras"

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = WORK_DIR / CHECKPOINT_NAME


def find_file(name):
    root = Path("/kaggle/input")
    hits = sorted(root.glob(f"**/{name}"))
    if not hits:
        raise FileNotFoundError(f"{name} introuvable — ajoutez-le au dataset Kaggle.")
    return hits[0]


def find_corpus_dir():
    npz = find_file("sleep_edf_corpus.npz")
    d = npz.parent
    if all((d / f).exists() for f in CORPUS_FILES):
        return d
    raise FileNotFoundError("Dataset corpus incomplet (3 fichiers requis).")


INPUT_DIR = find_corpus_dir()
NPZ_PATH = INPUT_DIR / "sleep_edf_corpus.npz"
META_PATH = INPUT_DIR / "sleep_edf_corpus_meta.json"
SPLIT_PATH = INPUT_DIR / "subject_split.json"

tf.random.set_seed(SEED)
np.random.seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

print("Corpus  :", INPUT_DIR)
print("Mode    :", "REPRISE" if RESUME else "DEPART ZERO")
print("Epochs  :", f"{INITIAL_EPOCH + 1} -> {EPOCHS}" if RESUME else f"1 -> {EPOCHS}")
print("GPU     :", tf.config.list_physical_devices("GPU"))

In [ ]:
def build_cnn_lstm_model(input_shape=(N_SAMPLES, 1), num_classes=5):
    model = Sequential([
        Conv1D(64, 3, activation="relu", input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.2),
        Conv1D(128, 3, activation="relu"),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),
        Bidirectional(LSTM(64, return_sequences=False)),
        Dropout(0.4),
        Dense(64, activation="relu"),
        Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def masks_from_manifest(subject_idx, subject_names, manifest):
    name_to_id = {name: i for i, name in enumerate(subject_names)}
    def _mask(names):
        ids = {name_to_id[n] for n in names if n in name_to_id}
        return np.isin(subject_idx, list(ids))
    return _mask(manifest["train_subjects"]), _mask(manifest["val_subjects"]), _mask(manifest["test_subjects"])


def compute_metrics(y_true, y_pred):
    labels = list(range(len(STAGE_NAMES)))
    report = classification_report(
        y_true, y_pred, labels=labels, target_names=STAGE_NAMES,
        output_dict=True, zero_division=0,
    )
    return {
        "n_samples": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0)),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred, labels=labels)),
        "per_class": {
            STAGE_NAMES[i]: {
                "f1": report[STAGE_NAMES[i]]["f1-score"],
                "recall": report[STAGE_NAMES[i]]["recall"],
                "support": int(report[STAGE_NAMES[i]]["support"]),
            }
            for i in labels
        },
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


class F1MacroCallback(tf.keras.callbacks.Callback):
    def __init__(self, X_val, y_val, patience=PATIENCE):
        super().__init__()
        self.X_val, self.y_val = X_val, y_val
        self.patience, self.best_f1, self.wait = patience, -1.0, 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_pred = np.argmax(self.model.predict(self.X_val, verbose=0), axis=1)
        macro = float(f1_score(self.y_val, y_pred, average="macro", zero_division=0))
        logs["val_f1_macro"] = macro
        print(f"  -> val_f1_macro = {macro:.4f}")
        if macro > self.best_f1 + 1e-4:
            self.best_f1, self.wait = macro, 0
        else:
            self.wait += 1
        if self.wait >= self.patience:
            print(f"  Early stop F1 macro (patience={self.patience})")
            self.model.stop_training = True

In [ ]:
print("Chargement du corpus...")
data = np.load(NPZ_PATH)
X = data["X"].astype(np.float32)
y = data["y"].astype(np.int32)
subject_idx = data["subject_idx"].astype(np.int32)
with open(META_PATH, encoding="utf-8") as f:
    subject_names = json.load(f)["subject_names"]
with open(SPLIT_PATH, encoding="utf-8") as f:
    manifest = json.load(f)
train_mask, val_mask, test_mask = masks_from_manifest(subject_idx, subject_names, manifest)
X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]
del data
print(f"Train {len(y_train):,} epoques · Val {len(y_val):,} · Test {len(y_test):,}")

In [ ]:
class_weights = dict(enumerate(class_weight.compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)))

if RESUME:
    ckpt = find_file(CHECKPOINT_NAME)
    model = tf.keras.models.load_model(ckpt)
    print("Checkpoint charge :", ckpt)
else:
    model = build_cnn_lstm_model(input_shape=(X.shape[1], 1))
    INITIAL_EPOCH = 0
    print("Nouveau modele")

model.summary()
callbacks = [
    F1MacroCallback(X_val, y_val),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(str(BEST_PATH), monitor="val_f1_macro", mode="max", save_best_only=True, verbose=1),
]

print(f"\n--- Entrainement epochs {INITIAL_EPOCH + 1} -> {EPOCHS} ---")
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    initial_epoch=INITIAL_EPOCH,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
    shuffle=True,
    verbose=2,
)

In [ ]:
if BEST_PATH.exists():
    model = tf.keras.models.load_model(BEST_PATH)
    print("Meilleur checkpoint :", BEST_PATH)

y_pred_val = np.argmax(model.predict(X_val, batch_size=BATCH_SIZE, verbose=0), axis=1)
y_pred_test = np.argmax(model.predict(X_test, batch_size=BATCH_SIZE, verbose=0), axis=1)
metrics_val = compute_metrics(y_val, y_pred_val)
metrics_test = compute_metrics(y_test, y_pred_test)

print("Validation : acc={:.4f}  F1={:.4f}  kappa={:.4f}".format(
    metrics_val["accuracy"], metrics_val["f1_macro"], metrics_val["cohen_kappa"]))
print("Test       : acc={:.4f}  F1={:.4f}  kappa={:.4f}".format(
    metrics_test["accuracy"], metrics_test["f1_macro"], metrics_test["cohen_kappa"]))
print("\nPar stade (test) :")
for name, row in metrics_test["per_class"].items():
    print(f"  {name}: F1={row['f1']:.3f}  recall={row['recall']:.3f}")

In [ ]:
model.save(WORK_DIR / "sleep_model_v1.keras")
with open(WORK_DIR / "metrics_val_bilstm.json", "w", encoding="utf-8") as f:
    json.dump(metrics_val, f, indent=2, ensure_ascii=False)
with open(WORK_DIR / "metrics_test_bilstm.json", "w", encoding="utf-8") as f:
    json.dump(metrics_test, f, indent=2, ensure_ascii=False)

print("Fichiers /kaggle/working/ :")
for p in sorted(WORK_DIR.glob("*")):
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")